# PaddleOCR and DeepSeek-OCR from existing layout regions

This Kaggle notebook saves two independent OCR outputs from the same existing layout sidecar. It does **not** rerun Chandra or a layout detector, and it does not use Chandra's text as OCR input. Start with the real page-34 smoke run; set `RUN_FULL_BOOK = True` only after it succeeds. Enable a Tesla T4 GPU.

In [ ]:
# Kaggle install cell. These are research-only dependencies; the starter runtime is unchanged.
%pip install -q "paddleocr>=3,<4" "paddlepaddle-gpu==3.2.0" -i https://www.paddlepaddle.org.cn/packages/stable/cu118/
%pip install -q "transformers==4.46.3" "tokenizers==0.20.3" sentencepiece einops easydict addict pymupdf pillow

In [ ]:
from pathlib import Path
import subprocess
import sys

# Clone the current main branch; do not use an old topic branch.
repo = Path('/kaggle/working/doc-agent-G07')
if not (repo / 'extras/ocr_research/kaggle-paddle-deepseek-ocr.py').is_file():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main',
                    'https://github.com/smammahdi/doc-agent-G07.git', str(repo)], check=True)
runner = repo / 'extras/ocr_research/kaggle-paddle-deepseek-ocr.py'
assert runner.is_file(), runner

# False = one real page first. True = all 1,034 pages after the smoke passes.
RUN_FULL_BOOK = False
LAYOUT_NAME = 'doclayout_yolo'  # change to 'chandra' for the Chandra regions
PAGES = 'all' if RUN_FULL_BOOK else '34'
OUTPUT = Path(f'/kaggle/working/paddle-deepseek-ocr-{LAYOUT_NAME}')
CACHE = Path(f'/kaggle/working/paddle-deepseek-ocr-cache-{LAYOUT_NAME}')
print({'runner': str(runner), 'layout': LAYOUT_NAME, 'pages': PAGES,
       'cuda': __import__('torch').cuda.is_available()})

In [ ]:
# The script discovers the Pierce PDF and layout sidecars under /kaggle/input.
# Outputs remain separate: paddleocr/ and deepseek-ocr/.
subprocess.run([
    sys.executable, str(runner),
    '--input-root', '/kaggle/input',
    '--layout-name', LAYOUT_NAME,
    '--pages', PAGES,
    '--engines', 'paddleocr,deepseek',
    '--output-root', str(OUTPUT),
    '--cache-root', str(CACHE),
    '--paddle-device', 'auto',
    '--deepseek-device', 'auto',
    '--deepseek-dtype', 'auto',
    '--deepseek-attention', 'eager',
], check=True)

In [ ]:
import json
import shutil

for engine_dir in (OUTPUT / 'paddleocr', OUTPUT / 'deepseek-ocr'):
    summary = engine_dir / 'summary.json'
    if summary.is_file():
        print(engine_dir.name, json.loads(summary.read_text()))

archive = shutil.make_archive(f'/kaggle/working/paddle-deepseek-ocr-results-{LAYOUT_NAME}', 'zip', OUTPUT)
print('download:', archive)